In [1]:
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import os
import re
import json
#from tqdm import tqdm
#import time
import os
from openai import OpenAI
import tiktoken

api_key = os.environ["OPENAI_API_KEY"]
org_key = os.environ.get("OPENAI_ORG_ID")

In [16]:
df_elig = pd.read_csv('paper_collection/WoS_251031_eligible_design.csv')
df_pgg = pd.read_csv('input/pgg_validation_basePrompt.csv')

In [ ]:
#df_elig['collected_251031'] =  df_elig['file_id'].isna() # get status (old or new)

In [17]:
df_elig['collected_251031'].sum()

640

In [ ]:
#df_elig.to_csv('paper_collection/WoS_251031_eligible.csv', index=False)

In [91]:
df_elig.sort_values('Times Cited, All Databases', ascending=False, ignore_index=True).to_csv('paper_collection/WoS_251031_eligible.csv', index=False)

In [90]:
df_elig.file_id.isna().sum()

0

In [121]:
for i in range(50, 800, 50):
    print(i)

50
100
150
200
250
300
350
400
450
500
550
600
650
700
750


In [12]:
client = OpenAI(api_key=api_key, organization=org_key) if org_key else OpenAI(api_key=api_key)

vector_store_id = "vs_68e867cb856881919afaf916060dcea8"

vector_store = client.vector_stores.retrieve(vector_store_id=vector_store_id)

In [13]:
vector_store

VectorStore(id='vs_68e867cb856881919afaf916060dcea8', created_at=1760061387, file_counts=FileCounts(cancelled=0, completed=757, failed=1, in_progress=0, total=758), last_active_at=1760101781, metadata={}, name='pgg_elig', object='vector_store', status='completed', usage_bytes=87378474, expires_after=None, expires_at=None, description=None)

Upload new file

In [ ]:
from openai import OpenAI
client = OpenAI()

client.files.create(
  file=open("mydata.jsonl", "rb"),
  purpose="fine-tune",
  expires_after={
    "anchor": "created_at",
    "seconds": 2592000
  }
)


In [15]:
file_upload = client.files.create(
  file=open(df_elig.iloc[0]['file_path'], "rb"),
  purpose='assistants'
)

In [16]:
file_upload

FileObject(id='file-6zCHAsvpPPD1vLzMz5TEWv', bytes=290564, created_at=1762626867, filename='10.1162_003355399556151.pdf', object='file', purpose='assistants', status='processed', expires_at=None, status_details=None)

In [40]:
file_upload.id

'file-Q3iDsVjchKvmfSut9qVSvn'

In [50]:
df_elig['collected_251031'].sum()

640

In [49]:
df_elig['file_id'].isna().sum()

640

In [41]:
for i, row in df_elig.iterrows():
    if row['collected_251031']:
        file_upload = client.files.create(
  file=open(row['file_path'], "rb"),
    purpose='assistants'
)
        row['file_id'] = file_upload.id # upload file id
        print(f"Upload: file_id {file_upload.id}")

Upload: file_id file-Kcy6VbJZZcPsmANjNttYiV
Upload: file_id file-Do8kuJ3RjSBJFQbqamBCs8
Upload: file_id file-7YdQByzV8bTcACBBF3oPx1
Upload: file_id file-AHz2ZAg7TvgqjTHN9goPRc
Upload: file_id file-Ejw5XG5diy7yxmEYnJqRhS
Upload: file_id file-6mQgzaRfs7ngwsuniLWMAg
Upload: file_id file-7RjAFMyTWKP9Wz6iSrzBEF
Upload: file_id file-A5jvWqMk99b3gp6LAoHAHg
Upload: file_id file-Toue5vF5Bt1RTG4h7qN9cy
Upload: file_id file-Nx5YMmX3CzbU4hx9X7tVGx
Upload: file_id file-8En1gfPxXisbqMSQXWBKSV
Upload: file_id file-H2S4QKMRNvnzHbjrHbiFW8
Upload: file_id file-MzDaGfdsFmXgfbfJ4eb4eK
Upload: file_id file-CTASPJZqBd1vUYmcN8bXLf
Upload: file_id file-Q4wsZwJgu9DfApt41XsvLT
Upload: file_id file-P3KNHt5qjLDq2V9UctEqjg
Upload: file_id file-RtQtL5NayMUPFC5fCxidPX
Upload: file_id file-4FJ5v1zCT2Kw9uq3prNtaw
Upload: file_id file-J2jdwA1ChB2CBHqsWwSGUv
Upload: file_id file-HB8NKneC5fVttFP4ymhMUN
Upload: file_id file-6xbJwXSMqWtDEfyZSNJiT5
Upload: file_id file-GQm9NuPg19ikAYqsj4ANi1
Upload: file_id file-GhK6i1wsd2Q

In [59]:
index_251031 = 0
for i, row in df_elig.iterrows():
    if row['collected_251031']:
        df_elig.loc[i, 'file_id'] = script.split("\nUpload: file_id ")[1:][index_251031]
        index_251031 += 1

In [62]:
df_elig.file_id.isna().sum()

0

In [65]:
file_batches = list()
for i, row in df_elig[df_elig['collected_251031']].iterrows():
    file_batches.append(
        {
            "file_id": row["file_id"],
            "attributes": {"fileID": row["file_id"]}
        }
    )

In [75]:
file_batches = list(df_elig[df_elig['collected_251031']]["file_id"])

In [77]:
len(file_batches)

640

In [68]:
def split_list_by_chunk_size(original_list, chunk_size=50):
    """
    Splits a list into sub-lists, each with a specified chunk size.

    Args:
        original_list (list): The list to be split.
        chunk_size (int): The maximum number of items in each sub-list.

    Returns:
        list: A list of sub-lists.
    """
    result_sublists = []
    for i in range(0, len(original_list), chunk_size):
        result_sublists.append(original_list[i:i + chunk_size])
    return result_sublists

In [79]:
for l in split_list_by_chunk_size(file_batches, 50):
    vector_store_file_batch = client.vector_stores.file_batches.create_and_poll(
  vector_store_id=vector_store_id,
  file_ids=l
)
    print(vector_store_file_batch)

    

VectorStoreFileBatch(id='vsfb_ed046d3737cd4fad84e4a7780cd1804a', created_at=1762631040, file_counts=FileCounts(cancelled=0, completed=50, failed=0, in_progress=0, total=50), object='vector_store.file_batch', status='completed', vector_store_id='vs_68e867cb856881919afaf916060dcea8')
VectorStoreFileBatch(id='vsfb_b80ae6c351bd4ba8b2a5e31e6df0300a', created_at=1762631046, file_counts=FileCounts(cancelled=0, completed=50, failed=0, in_progress=0, total=50), object='vector_store.file_batch', status='completed', vector_store_id='vs_68e867cb856881919afaf916060dcea8')
VectorStoreFileBatch(id='vsfb_5f5376606f4c47e78388e052917bbfe4', created_at=1762631050, file_counts=FileCounts(cancelled=0, completed=50, failed=0, in_progress=0, total=50), object='vector_store.file_batch', status='completed', vector_store_id='vs_68e867cb856881919afaf916060dcea8')
VectorStoreFileBatch(id='vsfb_bc2f4cef12ed45eaaa1b99841e3a2d3f', created_at=1762631176, file_counts=FileCounts(cancelled=0, completed=50, failed=0, in_

In [80]:
# update file attributes
for file in df_elig[df_elig['collected_251031']]["file_id"]:
    client.vector_stores.files.update(
        vector_store_id=vector_store_id,
        file_id=file,
        attributes={"fileId": file}
    )


### Update chunking strategy

In [84]:
# update file attributes
for file in df_elig["file_id"]:
    client.vector_stores.files.update(
        vector_store_id=vector_store_id,
        file_id=file,
        chunking_strategy={
        "type": "static",
        "max_chunk_size_tokens": 4096,
        "chunk_overlap_tokens": 500,
      }
    )


TypeError: Files.update() got an unexpected keyword argument 'chunking_strategy'

In [ ]:
vector_store_file_batch = client.vector_stores.file_batches.create(
  vector_store_id=vector_store_id,
  files=new_files
)
print(vector_store_file_batch)


In [67]:
len(file_batches)

640

In [ ]:
for i, row in df_elig.iterrows():
    if row['collected_251031']:
        client.vector_stores.files.create(
        vector_store_id=vector_store_id,
        file_id=row['file_id']
        )

In [ ]:
for fileid in script.split("\nUpload: file_id ")[1:]:
    

['file-Kcy6VbJZZcPsmANjNttYiV',
 'file-Do8kuJ3RjSBJFQbqamBCs8',
 'file-7YdQByzV8bTcACBBF3oPx1',
 'file-AHz2ZAg7TvgqjTHN9goPRc',
 'file-Ejw5XG5diy7yxmEYnJqRhS',
 'file-6mQgzaRfs7ngwsuniLWMAg',
 'file-7RjAFMyTWKP9Wz6iSrzBEF',
 'file-A5jvWqMk99b3gp6LAoHAHg',
 'file-Toue5vF5Bt1RTG4h7qN9cy',
 'file-Nx5YMmX3CzbU4hx9X7tVGx',
 'file-8En1gfPxXisbqMSQXWBKSV',
 'file-H2S4QKMRNvnzHbjrHbiFW8',
 'file-MzDaGfdsFmXgfbfJ4eb4eK',
 'file-CTASPJZqBd1vUYmcN8bXLf',
 'file-Q4wsZwJgu9DfApt41XsvLT',
 'file-P3KNHt5qjLDq2V9UctEqjg',
 'file-RtQtL5NayMUPFC5fCxidPX',
 'file-4FJ5v1zCT2Kw9uq3prNtaw',
 'file-J2jdwA1ChB2CBHqsWwSGUv',
 'file-HB8NKneC5fVttFP4ymhMUN',
 'file-6xbJwXSMqWtDEfyZSNJiT5',
 'file-GQm9NuPg19ikAYqsj4ANi1',
 'file-GhK6i1wsd2QjUMMT8ztdNp',
 'file-4A1Dp6xb8G1rua5tBXEpy8',
 'file-PhpNLK4kZHiUbhEts3uPMd',
 'file-6otzTJXUp9Ck2pdJ1UrDx8',
 'file-1WfZBoDFQE4dNSmbfooBcn',
 'file-5Mt4i8MBr6BQWV3oCNjdDq',
 'file-MHyjRX7qygCft3qqHmtV4P',
 'file-ArngoAN8oDsqmZMwRxnp41',
 'file-LX4cfiRCBNHUPtM2qM7bit',
 'file-6

In [45]:
df_elig['file_id'].isna().sum()

640

In [ ]:
# check upload status
for i, row in df_elig.iterrows():
    if row['collected_251031']:
        file = client.files.retrieve(
  row['file_id']
)
        print(file.status)
        #row['file_id'] = file_upload.id # upload file id
        #print(f"Upload: file_id {file_upload.id}")

In [ ]:
vector_store_file_batch = client.vector_stores.file_batches.create(
  vector_store_id=vector_store_id,
  files=new_files
)
print(vector_store_file_batch)


In [5]:
for file in df_elig.file_id:
    client.vector_stores.files.update(
        vector_store_id=vector_store_id,
        file_id=file,
        attributes={"fileId": file}
    )


In [9]:
client.vector_stores.files.list(
        vector_store_id=vector_store.id, limit=100
    ).data

[VectorStoreFile(id='file-v5st91VuWxMPIly2h4ZsE3qR', created_at=1760061695, last_error=None, object='vector_store.file', status='completed', usage_bytes=114006, vector_store_id='vs_68e867cb856881919afaf916060dcea8', attributes={'fileId': 'file-v5st91VuWxMPIly2h4ZsE3qR'}, chunking_strategy=StaticFileChunkingStrategyObject(static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static')),
 VectorStoreFile(id='file-csCPsOXyZ9eXygZoSZBGrwX6', created_at=1760061695, last_error=None, object='vector_store.file', status='completed', usage_bytes=176616, vector_store_id='vs_68e867cb856881919afaf916060dcea8', attributes={'fileId': 'file-csCPsOXyZ9eXygZoSZBGrwX6'}, chunking_strategy=StaticFileChunkingStrategyObject(static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static')),
 VectorStoreFile(id='file-HB7btOJ7vyoDV6lWr94fEWdO', created_at=1760061695, last_error=None, object='vector_store.file', status='completed', usa

In [4]:
vector_store

VectorStore(id='vs_68e867cb856881919afaf916060dcea8', created_at=1760061387, file_counts=FileCounts(cancelled=0, completed=757, failed=1, in_progress=0, total=758), last_active_at=1760062688, metadata={}, name='pgg_elig', object='vector_store', status='completed', usage_bytes=87378474, expires_after=None, expires_at=None, description=None)

In [178]:
vector_store_files = client.vector_stores.files.list(
        vector_store_id=vector_store.id, limit=100, filter='failed'
    )
for file_obj in vector_store_files.data:
        if file_obj.status == "failed":
            print({"file_id": file_obj.id, "filename": file_obj.id})

{'file_id': 'file-UV4X1AmP1CehypJPvbbdCicn', 'filename': 'file-UV4X1AmP1CehypJPvbbdCicn'}


In [ ]:
# note that this paper could not be parsed.
print(df_elig[df_elig.file_id == 'file-UV4X1AmP1CehypJPvbbdCicn'].file_path[55])

paper_collection/WoS_241106.Data/PDF/2044897762/10.2307_2297904.pdf


In [168]:
client.vector_stores.files.delete(
    vector_store_id=vector_store.id,
    file_id='file-UV4X1AmP1CehypJPvbbdCicn'
)

VectorStoreFileDeleted(id='file-UV4X1AmP1CehypJPvbbdCicn', deleted=True, object='vector_store.file.deleted')

In [173]:
file_batch = client.vector_stores.file_batches.create_and_poll(
vector_store_id=vector_store.id, file_ids=['file-UV4X1AmP1CehypJPvbbdCicn']
)

In [149]:
df_elig.shape

(758, 87)

In [124]:
file_batch = client.vector_stores.file_batches.create_and_poll(
vector_store_id=vector_store.id, file_ids=list(df_elig.file_id)[750:]
)

In [176]:
client.vector_stores.retrieve(vector_store_id=vector_store.id)

VectorStore(id='vs_68e867cb856881919afaf916060dcea8', created_at=1760061387, file_counts=FileCounts(cancelled=0, completed=757, failed=1, in_progress=0, total=758), last_active_at=1760062688, metadata={}, name='pgg_elig', object='vector_store', status='completed', usage_bytes=87378474, expires_after=None, expires_at=None, description=None)

In [93]:
client = OpenAI(api_key=api_key, organization=org_key) if org_key else OpenAI(api_key=api_key)
vector_stores = client.vector_stores.list(limit=100)

In [95]:
for store in vector_stores.data:
    print(f"Deleting vector store: {store.id}")
    deleted_store = client.vector_stores.delete(vector_store_id=store.id)
    print(f"Deleted: {deleted_store.id}")

Deleting vector store: vs_gQISmm7JBpwDhGBrlRPuhZd0
Deleted: vs_gQISmm7JBpwDhGBrlRPuhZd0
Deleting vector store: vs_1B4r8MfWjVyktWnJsbA2Db66
Deleted: vs_1B4r8MfWjVyktWnJsbA2Db66
Deleting vector store: vs_90F2TZBH5IwFoLjDsSl30Bgr
Deleted: vs_90F2TZBH5IwFoLjDsSl30Bgr
Deleting vector store: vs_Jk6B3RLTLxti4jXkPZeB00cA
Deleted: vs_Jk6B3RLTLxti4jXkPZeB00cA
Deleting vector store: vs_2D5euVCKtb0J6Sr0egDUO0B3
Deleted: vs_2D5euVCKtb0J6Sr0egDUO0B3
Deleting vector store: vs_FRs8nOfPIFzlnNbDkJDm0uaT
Deleted: vs_FRs8nOfPIFzlnNbDkJDm0uaT
Deleting vector store: vs_F6NI9XqPWGCjXCDdxCEzvzPb
Deleted: vs_F6NI9XqPWGCjXCDdxCEzvzPb
Deleting vector store: vs_EyFDfrcVVU3f7f8lHZ1QQfFr
Deleted: vs_EyFDfrcVVU3f7f8lHZ1QQfFr
Deleting vector store: vs_saUnFLuorHESoxf0wSutNuue
Deleted: vs_saUnFLuorHESoxf0wSutNuue
Deleting vector store: vs_vNM4HILUPdItl6qHBxolc0ps
Deleted: vs_vNM4HILUPdItl6qHBxolc0ps
Deleting vector store: vs_IJFfdO5IJdB4TOZILA5p8N9g
Deleted: vs_IJFfdO5IJdB4TOZILA5p8N9g
Deleting vector store: vs_ZXmwxI

In [6]:
PATH_MAP    = "paper_collection/collection_mapping_251110.json"          # {collection: [paper-row idx]}
with open(PATH_MAP, encoding="utf-8") as fh:
    raw_map = json.load(fh)

#n_rows = len(df_d)
coll_map = {lab: [int(x) for x in raw if str(x).isdigit() and 0 <= int(x)]
            for lab, raw in raw_map.items()}

# RAG

In [8]:
system_prompt = f"""We have conducted multiple public goods game experiments with varying experimental designs, to measure the effect of punishment in cooperative settings under various environments.
Your task is to predict how enabling a punishment mechanism to a specific game changes the ***efficiency*** compared to the same game with punishment disabled.
According to our experiments, whether punishment increases efficiency or not is highly dependent on a lot of dimensions in experiment design, and it is your job to navigate this heterogeneity and make accurate predictions.

***Efficiency*** is the ratio between the game players' behavior and that of a fully-cooperative group (i.e. a group in which all members contribute their full endowment in every round)
In other words, efficiency measures how close a group's total payoff is, compared to that of a group that always cooperates (i.e. always contributes the entire endowment, and benefits maximally from the multiplier). 
An efficiency value of 100% means that a group earned the same amount of coins as a hypothetical group that always cooperated.
            
For example, let's say a game has 5 players playing 10 rounds where 20 coins are given to each player per round and the multiplier for each contributed coin is 3.
In this case, the earning of a hypothetical "always cooperating" group is 5*10*20*3=3000 coins, while the earning of a hypothetical "never cooperating" group is 1000 coins.
Hence, the efficiency is 100% for the always cooperating group and 33% for the never cooperating group.

Your output should strictly be a prediction value with integer only (e.g., 33% should output 33 and nothing else)."""

def make_predict_prompt(config, augmented_text=''):
        closing =   f"""Now, predict the efficiency of the game below when punishment is to be enabled.
### Game Information ###
{config}

You predict that enabling punishment will cause the efficiency percentage to change to (output should be an integer and nothing else):
"""
        return augmented_text + closing
        
def make_config(cd): # cd: configuration dictionary
    game_structure = f"""
***The efficiency of this game with punishment disabled was:*** {int(round(100 * cd['efficiency_np'], 0))}%

[CONFIGURATION]

*** Game Structure ***
Number of players: {int(cd['CONFIG_playerCount'])}
Number of rounds: {int(cd['CONFIG_numRounds'])}
Is chat enabled among players?: {bool(cd['CONFIG_chat'])}
Is the contribution "all or nothing" i.e., binary instead of continuous?: {bool(cd['CONFIG_allOrNothing'])}
Is contribution the default i.e., does each player's endowment start in the public fund for them to opt-out?: {bool(cd['CONFIG_defaultContribProp'])}

*** Monetary Stakes ***
Marginal per capita return (MPCR): {cd['CONFIG_MPCR']}

*** Peer Incentives *** 
    """

    punishment = f"""
Punishment cost to impose a single unit of punishment: {int(cd['CONFIG_punishmentCost'])} coin(s)
Punishment impact (number of coins deducted from the punished player per coin spent punishing): {float(cd['CONFIG_punishmentTech'])}
    """
        # should it be punishment Tech? No magnitude is fine
    reward = f"""
Reward cost to grant a single unit of reward: {int(cd['CONFIG_rewardCost'])} coin(s)
Reward impact (the coins awarded to a player per coin spent rewarding): {float(cd['CONFIG_rewardTech'])}
    """

    information_display = f"""
*** Information Display ***
Is the number of rounds known to players (do they know when the game ends)?: {bool(cd['CONFIG_showNRounds'])}
Are peer outcomes shown (do players know how much their peers gained at the end of each round)?: {bool(cd['CONFIG_showOtherSummaries'])}"""

    information_punishment = f"""
When a player is punished/rewarded, are the punishers/rewarders known?: {bool(cd['CONFIG_showPunishmentId'])}
    """


    no_reward = f"""
Reward mechanism is not enabled.
    """

    
    if cd['CONFIG_rewardExists'] == True:
        return game_structure + punishment + reward + information_display + information_punishment
    else:
        return game_structure + punishment + no_reward + information_display + information_punishment

In [9]:
def append_prompt(dataframe, augmented_text=''):
    messages = [ make_predict_prompt( make_config(row), augmented_text=augmented_text ) for i, row in dataframe.iterrows() ]
    #dataframe['prompt_'+column] = messages
    return messages

prompt_rag = """You have access to the files of academic papers discussing the effect of punishment on cooperative games.
Make predictions based faithfully on what you can learn from the papers' integrated findings.
"""

prompts_rag = append_prompt(df_pgg, prompt_rag)

In [10]:
print(prompts_rag[0])

You have access to the files of academic papers discussing the effect of punishment on cooperative games.
Make predictions based faithfully on what you can learn from the papers' integrated findings.
Now, predict the efficiency of the game below when punishment is to be enabled.
### Game Information ###

***The efficiency of this game with punishment disabled was:*** 76%

[CONFIGURATION]

*** Game Structure ***
Number of players: 10
Number of rounds: 6
Is chat enabled among players?: True
Is the contribution "all or nothing" i.e., binary instead of continuous?: True
Is contribution the default i.e., does each player's endowment start in the public fund for them to opt-out?: False

*** Monetary Stakes ***
Marginal per capita return (MPCR): 0.62

*** Peer Incentives *** 
    
Punishment cost to impose a single unit of punishment: 2 coin(s)
Punishment impact (number of coins deducted from the punished player per coin spent punishing): 1.0
    
Reward mechanism is not enabled.
    
*** Inf

In [ ]:
def create_prediction_batch_json( model='gpt-4.1-2025-04-14', prompts = prompts_rag):
    requests = list()

    for n, p in enumerate(prompts):
        request = {
            "custom_id": f"ALL/Q{n+1}",
            "method": "POST",
            "url": "/v1/responses",
            "body": {
                "model": model,
                "input": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": p}
                ],
            "tools": [
                {
                    "type": "file_search",
                    "vector_store_ids": ["vs_68e867cb856881919afaf916060dcea8"],
                    "max_num_results": 50
                    }
                    ],
            "include": ["file_search_call.results", "message.output_text.logprobs"],
                "temperature": 0,
                #"logprobs": True,
                "top_logprobs": 20,
                #"max_output_tokens": 16
            }
        }
        requests.append(request)
    return requests


In [11]:
df_elig

,Unnamed: 0,Publication Type,Authors,Book Authors,Book Editors,Book Group Authors,Author Full Names,Book Author Full Names,Group Authors,Article Title,...,discipline.math,discipline.multi,discipline.bio,discipline.psych,discipline.social,jif.quartile,jif.value,type,design.efficiency,design.numVaried
0,0,J,"Fehr, E; Schmidt, KM",NaN,NaN,NaN,"Fehr, E; Schmidt, KM",NaN,NaN,"A theory of fairness, competition, and coopera...",...,False,False,False,False,False,Q1,11.1,theoretical,True,1
1,1,J,"Fehr, E; Gächter, S",NaN,NaN,NaN,"Fehr, E; Gächter, S",NaN,NaN,Altruistic punishment in humans,...,False,True,False,False,False,Q1,50.5,empirical,False,0
2,2,J,"Fehr, E; Gächter, S",NaN,NaN,NaN,"Fehr, E; Gächter, S",NaN,NaN,Cooperation and punishment in public goods exp...,...,False,False,False,False,False,Q1,10.5,empirical,True,0
3,3,J,"Ostrom, E",NaN,NaN,NaN,"Ostrom, E",NaN,NaN,Collective action and the evolution of social ...,...,False,False,False,False,False,Q1,6.9,review,True,1
4,4,J,"Herrmann, B; Thöni, C; Gächter, S",NaN,NaN,NaN,"Herrmann, Benedikt; Thoeni, Christian; Gachter...",NaN,NaN,Antisocial punishment across societies,...,False,True,False,False,False,Q1,44.7,empirical,True,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1393,1393,J,"Asgharpourmasouleh, A; Sadeghi, A; Yousofi, A",NaN,NaN,NaN,"Asgharpourmasouleh, Ahmadreza; Sadeghi, Atiye;...",NaN,NaN,A Grounded Agent-Based Model of Common Good Pr...,...,False,False,False,False,False,Q1,2.0,theoretical,False,1
1394,1394,J,"Myers, CD",NaN,NaN,NaN,"Myers, C. Daniel",NaN,NaN,Participation and punishment,...,False,False,False,False,True,Q3,0.6,empirical,False,0
1395,1395,J,"Spitzer, ML",NaN,NaN,NaN,"Spitzer, Matthew Laurence",NaN,NaN,Preferences over Punishment and Reward Mechani...,...,False,False,False,False,False,Q4,0.2,review,False,0
1396,1396,J,"Huang, YK; Inohara, T",NaN,NaN,NaN,"Huang, Yuankan; Inohara, Takehiro",NaN,NaN,Group-separations based on the repeated prison...,...,True,False,False,False,False,Q1,3.5,theoretical,False,0


In [18]:
df_elig['file_id']

0       file-u3msWQE8hahM5BeL9lXUkg4r
1       file-UCOHyTTkij32UXTFXLqubhI3
2       file-iJtIDqF8BnFoaU3ZpEFSQFTA
3       file-iEMtt1GM2k3nbvGNXpmzP8lH
4       file-baKy0FOC527QERo95AEoSeGX
                    ...              
1393      file-4LMSmSBT3ee4B1RBpZf8DV
1394      file-CBiNFuenCZ9LZhQD4ff79B
1395      file-LJM1kN2vC81jvSnPQciGnz
1396      file-QD57qdMLhnPLJrdgNvxdA9
1397      file-BeZBMmFMLr31irMT932yvN
Name: file_id, Length: 1398, dtype: object

In [12]:
def create_filtered_prediction_batch_json( model='gpt-4.1-2025-04-14', prompts = prompts_rag):
    requests = list()
    for k, v in coll_map.items():
        file_ids = df_elig['file_id'].iloc[v]  # file id list
        
        for n, p in enumerate(prompts):
            request = {
                "custom_id": f"{k}/Q{n+1}",
                "method": "POST",
                "url": "/v1/responses",
                "body": {
                    "model": model,
                    "input": [
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": p}
                    ],
                "tools": [
                    {
                        "type": "file_search",
                        "vector_store_ids": ["vs_68e867cb856881919afaf916060dcea8"],
                        "filters": {
                            "type": "in",
                            "key": "fileId",
                            "value": list(file_ids)
                        },
                        "max_num_results": 50
                        }
                        ],
                "include": ["file_search_call.results", "message.output_text.logprobs"],
                    "temperature": 0,
                    "top_logprobs": 20,
                }
            }
            requests.append(request)
    return requests


In [86]:
with open(f"OpenAI_batch_input/prediction_251108_RAG_41.jsonl", "w", encoding="utf-8") as jsonl_file:
    for i, entry in enumerate(create_prediction_batch_json()):
        jsonl_file.write(json.dumps(entry, ensure_ascii=False) + "\n")

In [19]:
with open(f"OpenAI_batch_input/prediction_251110_RAG_41.jsonl", "w", encoding="utf-8") as jsonl_file:
    for i, entry in enumerate(create_filtered_prediction_batch_json()):
        jsonl_file.write(json.dumps(entry, ensure_ascii=False) + "\n")